# Text Generation - Summarization

- https://github.com/ExamProCo/AWS-Examples/tree/main/bedrock/text-generation
- https://catalog.us-east-1.prod.workshops.aws/amazon-bedrock/en-US/110-text-generation
    - https://github.com/aws-samples/amazon-bedrock-workshop/blob/main/01_Text_generation/01_text_and_code_generation_w_bedrock.ipynb

- Video Tutorial: https://youtu.be/WZeZZ8_W-M4?si=9-FdcilXTwd4lK-u&t=12490

## Requirements

1. Request access to Bedrock
2. Enable models to be used

## Set Up

**Libraries**

In [ ]:
%pip install boto3

In [ ]:
import json
import boto3
import botocore
from IPython.display import display, Markdown
import time

**Client & Global Variables**
<!-- <div class="alert alert-block alert-info"> </div> -->
<!-- <div class="alert alert-block alert-warning"></div> -->

In [ ]:
# Initialize Bedrock client
# session = boto3.session.Session()
# region = session.region_name
# client = boto3.client(service_name='bedrock-runtime', region_name=region)

client = boto3.client(service_name='bedrock-runtime')

## Description

- **Define text to summarize** - will be used by 2 APIs to summarize same text.
- **API Call:**
    1. Create request body (includes prompt data, other configs)
    2. Send request
    3. Get response to read body.
    4. Extract answer from body.
- APIs
    - **Invoke Modle API:** model-specific reques format
    - **Converse API:** unified interface that can be used accross different models using the exact same request format.


In [ ]:
text_to_summarize = """
AWS took all of that feedback from customers, and today we are excited to announce Amazon Bedrock, \
a new service that makes FMs from AI21 Labs, Anthropic, Stability AI, and Amazon accessible via an API. \
Bedrock is the easiest way for customers to build and scale generative AI-based applications using FMs, \
democratizing access for all builders. Bedrock will offer the ability to access a range of powerful FMs \
for text and images—including Amazons Titan FMs, which consist of two new LLMs we're also announcing \
today—through a scalable, reliable, and secure AWS managed service. With Bedrock's serverless experience, \
customers can easily find the right model for what they're trying to get done, get started quickly, privately \
customize FMs with their own data, and easily integrate and deploy them into their applications using the AWS \
tools and capabilities they are familiar with, without having to manage any infrastructure (including integrations \
with Amazon SageMaker ML features like Experiments to test different models and Pipelines to manage their FMs at scale).
"""

## APIs
- Invoke Model API
- Converse API

### Text Summarization - **Invoke Model API**
- model: amazon titan

In [ ]:
prompt_data = """Please provide a summary of the following text. Do not add any information that is not mentioned in the text below.
<text>
{text_to_summarize}
</text>

"""

In [ ]:
body = json.dumps({
    "inputText": prompt_data,
    "textGenerationConfig": {
        "maxTokenCount": 1024,
        "stopSequences": [],
        "temperature": 0,
        "topP": 1,
    }
})

In [ ]:
response = bedrock.invoke_model(
    modelId="anthropic.claude-2",
    accept="application/json",
    contentType="application/json",
    body=body
)

In [ ]:
body = json.loads(response.get("body").read())
answer = body.get("results")[0].get("outputText")
print("Answer:\n", answer)


### Text Summarization - **Converse API**

> Unified interface for different models.

#### **Converse API Request Payload**

```json
{
  "modelId": <Required (str)>,  
  "messages": <Required: Conversation history (List[dict])>,
  "system":  <Optional: System instructions (List[dict])>,
  "inferenceConfig": <Optional: Inference parameters (dict)>,
  "toolConfig":  <Optional: Function calling setup (dict) >,
}
```

##### **Messages (Required)**

```json
[
    {
        "role": "",     //sender
        "content": [
            {
            "text": "", //prompt/message content
            }
        ],
    }
]
```

##### **System Instructions**

```json
[
    {
      "text": ""
    }
]
```

##### **Configuration Parameters**

```json
{
    "temperature": 0.7, // Randomness (0.0-1.0)
    "topP": 0.9,        // Diversity control (0.0-1.0)
    "maxTokens": 2000,  // Maximum response length
    "stopSequences": [] // Stop generation triggers  
}
```

##### **Function Calling Setup**
```json
{
    "tools": [],
    "toolChoice": {
      "auto": {}        // Let model decide when to use tools
    }
}

#### **Converse API Response**

##### Extract model's answer

```python
answer = response["output"]["message"]["content"][0]["text"]
```

In [ ]:
prompt_data = """Please provide a summary of the following text. Text to summarize: {text_to_summarize}"""

In [ ]:
message = [
    {
        "role": "user",
        "content": [
            {
            "text": prompt_data,
            }
        ],
    }
]

params = {
    "temperature": 0.4,
    "topP": 0.9,
    "maxTokens": 500,    
}

In [ ]:
response = client.converse(
        modelId ="amazon.titan-tg1-large",
        messages = message,
        inferenceConfig = params,
    )

In [ ]:
answer = response["output"]["message"]["content"][0]["text"]
display_response(answer, "Claude 3.7 Sonnet (Converse API)")


***